In [12]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [13]:
surnames = open("../data/surnames_hy.txt", "r", encoding="utf-8").read().splitlines()
surnames[:3]

['աբազյան', 'աբալակով', 'աբալյան']

In [14]:
len(surnames)

689

In [15]:
chars = sorted(list(set(''.join(surnames))))
chars[:5]

['ա', 'բ', 'գ', 'դ', 'ե']

In [16]:
dataset_len = len(chars)

In [17]:
char_to_num = {s:i+1 for i,s in enumerate(chars)}
char_to_num['.'] = 0
num_to_char = {i:s for s,i in char_to_num.items()}
print(num_to_char)
print(char_to_num)

{1: 'ա', 2: 'բ', 3: 'գ', 4: 'դ', 5: 'ե', 6: 'զ', 7: 'է', 8: 'թ', 9: 'ժ', 10: 'ի', 11: 'լ', 12: 'խ', 13: 'ծ', 14: 'կ', 15: 'հ', 16: 'ձ', 17: 'ղ', 18: 'ճ', 19: 'մ', 20: 'յ', 21: 'ն', 22: 'շ', 23: 'ո', 24: 'չ', 25: 'պ', 26: 'ջ', 27: 'ռ', 28: 'ս', 29: 'վ', 30: 'տ', 31: 'ր', 32: 'ց', 33: 'ւ', 34: 'փ', 35: 'ք', 36: 'օ', 37: 'ֆ', 38: 'և', 0: '.'}
{'ա': 1, 'բ': 2, 'գ': 3, 'դ': 4, 'ե': 5, 'զ': 6, 'է': 7, 'թ': 8, 'ժ': 9, 'ի': 10, 'լ': 11, 'խ': 12, 'ծ': 13, 'կ': 14, 'հ': 15, 'ձ': 16, 'ղ': 17, 'ճ': 18, 'մ': 19, 'յ': 20, 'ն': 21, 'շ': 22, 'ո': 23, 'չ': 24, 'պ': 25, 'ջ': 26, 'ռ': 27, 'ս': 28, 'վ': 29, 'տ': 30, 'ր': 31, 'ց': 32, 'ւ': 33, 'փ': 34, 'ք': 35, 'օ': 36, 'ֆ': 37, 'և': 38, '.': 0}


In [18]:
window_size = 3
X, Y = [], []

for surname in surnames:
    context = [0] * window_size
    for char in surname + '.':
        char_index = char_to_num[char]
        X.append(context)
        Y.append(char_index)

        print(''.join(num_to_char[i] for i in context), '--->', num_to_char[char_index])

        # In general if window size > 1
        context = context[1:] + [char_index]
        #context = [char_index]

... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> զ
բազ ---> յ
ազյ ---> ա
զյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> լ
բալ ---> ա
ալա ---> կ
լակ ---> ո
ակո ---> վ
կով ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> լ
բալ ---> յ
ալյ ---> ա
լյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> յ
բայ ---> ա
այա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ջ
բաջ ---> յ
աջյ ---> ա
ջյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ս
բաս ---> յ
ասյ ---> ա
սյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ր
բար ---> դ
արդ ---> յ
րդյ ---> ա
դյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ր
բար ---> յ
արյ ---> ա
րյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ր
բար ---> տ
արտ ---> յ
րտյ ---> ա
տյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> գ
աբգ ---> ա
բգա ---> ր
գար ---> յ
արյ ---> ա
րյա ---> ն
յան ---> .
... ---> ա
..ա ---> բ
.աբ ---> դ
աբդ ---> ա
բդա ---> լ
դալ ---> յ
ալյ ---> ա
լյա ---> ն

In [19]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [20]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([6530, 3]), torch.int64, torch.Size([6530]), torch.int64)

In [21]:
def build_dataset(surnames):
    X, Y = [], []
    for surname in surnames:
        context = [0] * window_size
        for char in surname + '.':
            char_index = char_to_num[char]
            X.append(context)
            Y.append(char_index)

            print(''.join(num_to_char[i] for i in context), '--->', num_to_char[char_index])

            # In general if window size > 1
            context = context[1:] + [char_index]
            #context = [char_index]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

In [22]:
import random
SEED = 42
random.seed(SEED)

In [23]:
random.shuffle(surnames)
n1 = int(0.8*len(surnames))
n2 = int(0.9*len(surnames))

In [24]:
X_train, Y_train = build_dataset(surnames[:n1])
X_valid, Y_valid = build_dataset(surnames[n1:n2])
X_test, Y_test = build_dataset(surnames[n2:])

... ---> ա
..ա ---> բ
.աբ ---> ա
աբա ---> ր
բար ---> տ
արտ ---> յ
րտյ ---> ա
տյա ---> ն
յան ---> .
... ---> թ
..թ ---> ո
.թո ---> վ
թով ---> մ
ովմ ---> ա
վմա ---> ս
մաս ---> յ
ասյ ---> ա
սյա ---> ն
յան ---> .
... ---> գ
..գ ---> ո
.գո ---> ր
գոր ---> գ
որգ ---> ի
րգի ---> ս
գիս ---> յ
իսյ ---> ա
սյա ---> ն
յան ---> .
... ---> ճ
..ճ ---> գ
.ճգ ---> ն
ճգն ---> ա
գնա ---> վ
նավ ---> ո
ավո ---> ր
վոր ---> յ
որյ ---> ա
րյա ---> ն
յան ---> .
... ---> շ
..շ ---> ա
.շա ---> թ
շաթ ---> ի
աթի ---> ր
թիր ---> յ
իրյ ---> ա
րյա ---> ն
յան ---> .
... ---> բ
..բ ---> ա
.բա ---> բ
բաբ ---> լ
աբլ ---> ո
բլո ---> յ
լոյ ---> ա
ոյա ---> ն
յան ---> .
... ---> մ
..մ ---> ի
.մի ---> ր
միր ---> ի
իրի ---> մ
րիմ ---> ա
իմա ---> ն
ման ---> յ
անյ ---> ա
նյա ---> ն
յան ---> .
... ---> մ
..մ ---> ե
.մե ---> լ
մել ---> ի
ելի ---> ք
լիք ---> ս
իքս ---> ե
քսե ---> թ
սեթ ---> յ
եթյ ---> ա
թյա ---> ն
յան ---> .
... ---> ս
..ս ---> ա
.սա ---> մ
սամ ---> վ
ամվ ---> ե
մվե ---> լ
վել ---> յ
ելյ ---> ա
լյա ---> ն
յան ---> .

In [25]:
embedding_dim = 16
vocab_size = dataset_len + 1
input_size = window_size * embedding_dim
hidden_neurons = 200

g = torch.Generator().manual_seed(SEED)
C = torch.randn((vocab_size, embedding_dim), generator=g, requires_grad=True)
W1= torch.randn((input_size, hidden_neurons), generator=g, requires_grad=True)
b1 = torch.randn(hidden_neurons, generator=g, requires_grad=True)
W2 = torch.randn((hidden_neurons, vocab_size), generator=g, requires_grad=True)
b2 = torch.randn(vocab_size, generator=g, requires_grad=True)

parameters = [C, W1, b1, W2, b2]


In [26]:
# Number of parameters
sum(p.nelement() for p in parameters)

18263

In [27]:
batch_size = 32
num_epochs = 10
learning_rate = 0.1

In [28]:
lossi = []

In [29]:
X_train.shape

torch.Size([5225, 3])

Should we do update for each batch or for epock?

In [30]:
epoch_loss = 0
for epoch in range(num_epochs):
    permutation = torch.randperm(X_train.shape[0])

    X_train_shuffled = X_train[permutation]
    Y_train_shuffled = Y_train[permutation]

    for start in range(0, X_train.shape[0], batch_size):
        end = start + batch_size

        X_batch = X_train_shuffled[start:end]
        Y_batch = Y_train_shuffled[start:end]

        # Forward pass
        emb = C[X_batch]
        h = torch.tanh(emb.view(-1, input_size) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Y_batch)

        # Backward pass
        for p in parameters:
            p.grad = None

        loss.backward()

        # Update
        for p in parameters:
            p.data += -learning_rate * p.grad

    epoch_loss += loss.item()

    epoch_loss /= len(range(0, X_train.shape[0], batch_size))

    lossi.append(epoch_loss)

    print(f"Epoch {epoch}: loss = {epoch_loss:.4f}")
    

Epoch 0: loss = 0.0491
Epoch 1: loss = 0.0379
Epoch 2: loss = 0.0080
Epoch 3: loss = 0.0059
Epoch 4: loss = 0.0148
Epoch 5: loss = 0.0177
Epoch 6: loss = 0.0092
Epoch 7: loss = 0.0112
Epoch 8: loss = 0.0099
Epoch 9: loss = 0.0039


In [34]:
with torch.no_grad():
    emb = C[X_train]
    h = torch.tanh(emb.view(-1, input_size) @ W1 + b1)
    logits = h @ W2 + b2
    val_loss = F.cross_entropy(logits, Y_train)

print("Train loss:", val_loss.item())

Train loss: 1.7617871761322021


In [32]:
with torch.no_grad():
    emb = C[X_valid]
    h = torch.tanh(emb.view(-1, input_size) @ W1 + b1)
    logits = h @ W2 + b2
    val_loss = F.cross_entropy(logits, Y_valid)

print("Validation loss:", val_loss.item())

Validation loss: 3.83109974861145


In [37]:
# sample from the model
g = torch.Generator().manual_seed(SEED)

for _ in range(20):
    
    out = []
    context = [0] * window_size
    while True:
      emb = C[torch.tensor([context])]
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(num_to_char[i] for i in out))

գյունց.
չիվիկյան.
ջույուլսոյան.
զասդրշչյան.
հուդալյան.
եղրյան.
բաբուդածյան.
գելունի.
թումյան.
ալալյան.
խավաջյան.
եսոեօասևմֆգզագղոչրունուկյան.
ննվելյան.
հեքիսյան.
ղույումյան.
գուլեֆտանյան.
եսոեօասևմֆգզագյան.
հունի.
թելյան.
օակյան.
